# YOLOv11 Multi-GPU Training on HA11 Dataset
This notebook is designed to be executed on **Kaggle** using the **Dual T4 GPU** environment.

## Prerequisites
1. Enable the **T4 x2** accelerator in the Kaggle notebook settings.
2. Attach the dataset `ha11` to this notebook (which will map to `/kaggle/input/ha11-v1-ha11-yolov11` or similar).

In [ ]:
# 1. Install Dependencies
!pip install ultralytics -q
import ultralytics
from ultralytics import YOLO
import os
import yaml

ultralytics.checks()

## 2. Dataset Path Configuration
Since the dataset is already in YOLO format, we just need to locate the `data.yaml` file dynamically and update its internal paths to point to the Kaggle input directory.

In [ ]:
KAGGLE_WORKING_DIR = "/kaggle/working"
YAML_PATH = None

# Dynamically find the data.yaml file
for root, dirs, files in os.walk('/kaggle/input'):
    if 'data.yaml' in files:
        YAML_PATH = os.path.join(root, 'data.yaml')
        break

if YAML_PATH:
    print(f"Found dataset YAML at: {YAML_PATH}")
    
    # Read the YAML to parse its contents
    with open(YAML_PATH, 'r') as f:
        data_yaml = yaml.safe_load(f)
    
    # Update paths to absolute paths so YOLO finds them
    base_dir = os.path.dirname(YAML_PATH)
    
    # Some YOLO datasets define paths relative to the yaml file, some have absolute paths broken by Kaggle.
    # The safest way is to overwrite the 'path' attribute.
    data_yaml['path'] = base_dir
    
    # Make sure train/val/test are relative to 'path'
    if 'train' in data_yaml and data_yaml['train'].startswith(base_dir):
        data_yaml['train'] = data_yaml['train'].replace(base_dir + '/', '')
    if 'val' in data_yaml and data_yaml['val'].startswith(base_dir):
        data_yaml['val'] = data_yaml['val'].replace(base_dir + '/', '')
    
    # Write the corrected YAML to the working directory
    corrected_yaml_path = os.path.join(KAGGLE_WORKING_DIR, "corrected_data.yaml")
    with open(corrected_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f)
        
    print(f"Corrected YAML saved to: {corrected_yaml_path}")
else:
    print("Could not find data.yaml. Please ensure the dataset is attached.")

## 3. Multi-GPU Training with Luminosity-Only Augmentation
We configure the YOLO engine to disable all spatial and color augmentations **except** for `hsv_v` (Value/Luminosity). This is critical for datasets where exact shape/orientation matters but lighting conditions vary.

In [ ]:
# Initialize the YOLOv11 nano model
model = YOLO('yolo11n.pt')

# Train the model with specific augmentations disabled
results = model.train(
    data=corrected_yaml_path,
    epochs=50,
    imgsz=640,
    batch=64,           # Dual T4 GPUs can handle 64 easily for nano
    device=[0, 1],      # <--- ENABLE DataParallel (Uses both T4 GPUs)
    workers=4,
    project='/kaggle/working/ha11_yolo',
    name='luminosity_only_run',
    
    # --- LUMINOSITY ONLY AUGMENTATION ---
    hsv_h=0.0,          # Disable Hue
    hsv_s=0.0,          # Disable Saturation
    hsv_v=0.4,          # Enable Value (Luminosity) variation (Default YOLO value is 0.4)
    degrees=0.0,        # Disable Rotation
    translate=0.0,      # Disable Translation
    scale=0.0,          # Disable Scaling
    shear=0.0,          # Disable Shear
    perspective=0.0,    # Disable Perspective
    flipud=0.0,         # Disable Flip Up-Down
    fliplr=0.0,         # Disable Flip Left-Right
    mosaic=0.0,         # Disable Mosaic
    mixup=0.0,          # Disable MixUp
    copy_paste=0.0      # Disable Copy-Paste
)

In [ ]:
# 4. Zip and Save Results
!zip -r /kaggle/working/ha11_training_results.zip /kaggle/working/ha11_yolo/luminosity_only_run
print("Training complete! Download ha11_training_results.zip from the output pane.")

## 5. Illustration of Model Results using Graphs
We can display the automatically generated training convergence graphs and confusion matrix inline.

In [ ]:
from IPython.display import Image, display
import os

run_dir = "/kaggle/working/ha11_yolo/luminosity_only_run"

# Display training loss curves and mAP graphs
results_img = os.path.join(run_dir, "results.png")
if os.path.exists(results_img):
    print("Training Convergence & Metrics:")
    display(Image(filename=results_img, width=1000))

# Display the Confusion Matrix
cm_img = os.path.join(run_dir, "confusion_matrix.png")
if os.path.exists(cm_img):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_img, width=800))

# Display sample validation predictions
val_pred_img = os.path.join(run_dir, "val_batch0_pred.jpg")
if os.path.exists(val_pred_img):
    print("\nValidation Predictions:")
    display(Image(filename=val_pred_img, width=1000))
